# Задание 7.1: Волновое уравнение

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import math
import matplotlib.animation as anim
from IPython.display import HTML
from scipy.linalg import solve_banded

# Параметры

In [ ]:
L = 1 # длина струны
c = 1 # параметр уравнения

t0 = 0 # нач момент
t1 = 2 # конечный момент

dt = 0.001 # шаг по времени

# Параметры моделирования

In [ ]:
Nx = 101 # кол-во вершин

dx = L/(Nx-1) # шаг по пространству

prc = 10/100 # какую часть струны занимает треугольный профиль

u0 = np.zeros(Nx)
count_node = math.floor(prc*(Nx-1)/2)
mid_pnt = math.floor((Nx-1)/2)
u0[mid_pnt:mid_pnt+count_node+1] = np.linspace(1,0,count_node+1)
u0[mid_pnt-count_node:mid_pnt+1] = np.linspace(0,1,count_node+1)

du0 = np.zeros(Nx)

x = np.linspace(0,L,Nx)

# Параметры анимации

In [ ]:
anim_time = 5 # время на анимацию
fps = 20 # кол-во кадров в ссекунду

# Схема "Крест"

In [ ]:
def krest( u0 , du0 , c , dx , dt , t0 , t1 ):
    Nx = len(u0) # кол-во вершин
    Nt = math.floor((t1-t0)/dt) #кол-во шагов равных dt
    dt_last = t1 - t0 - Nt*dt # последний шаг, который может быть меньше, так как надо точно попасть в t1

    print(f"CFL : {dt**2*c**2/dx**2}") # число Куранта

    if math.fabs(dt_last) > 1.e-8:
        u = np.zeros( ( 1+Nt+1 , Nx ) )
    else:
        u = np.zeros( ( 1+Nt , Nx ) )
    
    u[0,:] = u0
    if Nt > 0 :
        u[1,:] = u[0,:] + dt*du0
    else:
        u[1,:] = u[0,:] + dt_last*du0

    for j in range(1,Nt):
        # if j % 100 == 0:
        #     print(f"{j}/{Nt}")
        u[j+1,1:Nx-1] = 2*u[j,1:Nx-1] - u[j-1,1:Nx-1] + dt**2 * ( c**2 * ( u[j,0:Nx-2] - 2*u[j,1:Nx-1] + u[j,2:Nx] )/dx**2 )
        u[j+1,0] = 0
        u[j+1,-1] = 0
    
    if math.fabs(dt_last) > 1.e-8:
        j = Nt
        u[j+1,1:Nx-1] = 2*u[j,1:Nx-1] - u[j-1,1:Nx-1] + dt**2 * ( c**2 * ( u[j,0:Nx-2] - 2*u[j,1:Nx-1] + u[j,2:Nx] )/dx**2 )
        u[j+1,0] = 0
        u[j+1,-1] = 0
        
    return u

# Схема "Крест" , но вместо задания шага по времени задается CFL

In [ ]:
def krest_cfl( u0 , du0 , c , dx , cfl , t0 , t1 ):
    dt = math.sqrt( cfl * dx**2 / c**2 )
    return krest( u0 , du0 , c , dx , dt , t0 , t1 )

In [ ]:
def animat( u0 , du0 , c , dx , cfl , t0 , t1 ):
    u = krest_cfl( u0 , du0 , c , dx , cfl , t0 , t1)
    step = 1 + math.floor(len(u)/(fps*anim_time))
    u = u[::step,]
    #plt.plot(x,u[0,:],color="blue")
    #plt.plot(x,u[-1,:],color="red")
    #plt.show()
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    line, = ax.plot(x, u[0, :], 'b-', linewidth=2)
    ax.set_xlim(0, L)
    ax.set_ylim(u.min(), u.max())
    ax.set_xlabel('Позиция x')
    ax.set_ylabel('Значение u')
    ax.set_title(f'Анимация распространения волны, CFL = {cfl}')
    ax.grid(True)
    
    def uupdate(frame):
        line.set_ydata(u[frame, :])
        return line,
    
    # Создаем анимацию
    animt = anim.FuncAnimation(
        fig, 
        uupdate, 
        frames=len(u),
        interval=1000/fps,  # интервал между кадрами в миллисекундах
        blit=True,
        repeat=True
    )
    
    # Отображаем анимацию в Jupyter
    plt.close()  # Скрываем статичный график
    return HTML(animt.to_jshtml())

In [ ]:
animat( u0 , du0 , c , dx , dt**2 * c**2 / dx**2 , t0 , t1)

In [ ]:
animat( u0 , du0 , c , dx , 0.5 , t0 , t1)

In [ ]:
animat( u0 , du0 , c , dx , 0.2 , t0 , t1)

In [ ]:
animat( u0 , du0 , c , dx , 1 , t0 , t1)

### При CFL большем 1 видно, что cхема разваливается при таких начальных и граничных уловиях. Однако при 1 устойчивость сохраняется.

In [ ]:
animat( u0 , du0 , c , dx , 1.001 , t0 , t1)